In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
data = np.load("data/train/train_signals.npz")

In [8]:
data.keys()

KeysView(NpzFile 'data/train/train_signals.npz' with keys: flight_id, signals, channel_names)

In [ ]:
data["signals"]

(15872, 128, 8)

In [ ]:
pd.DataFrame.from_dict(
    {name: channel[0] for channel, name in zip(np.transpose(data["signals"], axes=(2,0,1)), data["channel_names"])}
)

,accel_x,accel_y,accel_z,gyro_z,motor_current_1,motor_current_2,vibration,battery_voltage
0,-0.065681,0.003845,0.088945,0.091702,0.068763,0.039613,0.037937,0.908622
1,-0.004804,0.036983,0.031530,-0.057574,0.082236,0.049392,0.020964,1.004656
2,-0.048994,-0.082252,0.034755,-0.054014,0.055887,-0.000031,0.039784,1.069685
3,0.020671,-0.052529,0.083901,-0.011660,-0.000663,0.000578,-0.004961,0.947578
4,-0.065176,0.051167,-0.012283,-0.058412,0.092972,-0.111027,-0.000654,0.991390
...,...,...,...,...,...,...,...,...
123,0.130730,0.056980,0.097268,-0.004616,-0.003164,0.042306,0.072186,1.042322
124,0.106082,0.121909,0.146401,-0.002869,0.084981,0.059206,-0.040773,0.999129
125,0.068434,0.162466,0.175543,0.070207,0.102734,-0.052855,-0.055226,0.995077
126,0.113677,0.022890,0.127077,-0.033728,0.040412,0.064584,-0.016325,0.992999


In [39]:
data["flight_id"]

array([    0,     1,     2, ..., 21031, 21032, 21033], shape=(15872,))

In [ ]:
X = data["signals"].transpose(0, 2, 1)
flight_ids = data["flight_id"]

# Load labels
labels_df = pd.read_csv("data/train/train_tabular.csv", usecols=["flight_id", "failure_within_horizon"]) 

# Join
sensor_df = pd.DataFrame({"flight_id": flight_ids, "array_idx": np.arange(len(flight_ids))})
merged = sensor_df.merge(labels_df, on="flight_id", how="inner")

# Pull out aligned arrays
X_aligned = X[merged["array_idx"].values]   # (M, 8, 128)
y_aligned = merged["failure_within_horizon"].values           # (M,)
flight_ids_aligned = merged["flight_id"].values

In [45]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

# --- 1. Dataset ---
class DroneDataset(Dataset):
    def __init__(self, data, labels, normalize=True):
        # data: numpy array of shape (N, 8, 128)
        # labels: numpy array of shape (N,) — 1 = will crash, 0 = safe
        self.X = torch.tensor(data, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.float32)
        
        if normalize:
            # Normalize per channel across the dataset
            mean = self.X.mean(dim=(0, 2), keepdim=True)  # (1, 8, 1)
            std  = self.X.std(dim=(0, 2), keepdim=True) + 1e-8
            self.X = (self.X - mean) / std

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# --- 2. Model ---
class DroneCrashClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(8, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # (B, 32, 64)

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),                          # (B, 64, 32)

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),                  # (B, 128, 1)
        )
        self.head = nn.Sequential(
            nn.Flatten(),                             # (B, 128)
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.head(self.encoder(x)).squeeze(-1)


# --- 3. Training loop ---
def train(model, loader, optimizer, criterion, device):
    model.train()
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()


device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = DroneCrashClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# Use pos_weight if crashes are rare — e.g. 10× more safe flights than crashes
criterion = nn.BCELoss()  # or BCEWithLogitsLoss + remove Sigmoid above

In [48]:
from sklearn.metrics import average_precision_score

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            preds = model(X_batch)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y_batch.numpy())
    
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    
    return average_precision_score(all_labels, all_preds)

In [51]:
from torch.utils.data import random_split

# 1. Create the full dataset
dataset = DroneDataset(X_aligned, y_aligned)

# 2. Split into train and val
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# 3. Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

# 4. Instantiate model, loss, optimizer
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = DroneCrashClassifier().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

# 5. Training loop
for epoch in range(100):
    # Training
    model.train()
    train_losses = []
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    # Validation
    model.eval()
    val_losses, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            val_losses.append(criterion(preds, y_batch).item())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(y_batch.cpu().numpy())

    auprc = average_precision_score(
        np.concatenate(all_labels),
        np.concatenate(all_preds)
    )
    print(f"Epoch {epoch+1:02d} — train loss: {np.mean(train_losses):.4f} | val loss: {np.mean(val_losses):.4f} | val AUPRC: {auprc:.4f}")

Epoch 01 — train loss: 0.3676 | val loss: 0.3511 | val AUPRC: 0.2617
Epoch 02 — train loss: 0.3558 | val loss: 0.3459 | val AUPRC: 0.2679
Epoch 03 — train loss: 0.3509 | val loss: 0.3481 | val AUPRC: 0.2797
Epoch 04 — train loss: 0.3508 | val loss: 0.3420 | val AUPRC: 0.2807
Epoch 05 — train loss: 0.3474 | val loss: 0.3451 | val AUPRC: 0.2870
Epoch 06 — train loss: 0.3437 | val loss: 0.3436 | val AUPRC: 0.2864
Epoch 07 — train loss: 0.3442 | val loss: 0.3416 | val AUPRC: 0.2719
Epoch 08 — train loss: 0.3458 | val loss: 0.3435 | val AUPRC: 0.2869
Epoch 09 — train loss: 0.3449 | val loss: 0.3419 | val AUPRC: 0.2924
Epoch 10 — train loss: 0.3441 | val loss: 0.3490 | val AUPRC: 0.2960
Epoch 11 — train loss: 0.3450 | val loss: 0.3423 | val AUPRC: 0.2920
Epoch 12 — train loss: 0.3430 | val loss: 0.3426 | val AUPRC: 0.2948
Epoch 13 — train loss: 0.3441 | val loss: 0.3422 | val AUPRC: 0.2957
Epoch 14 — train loss: 0.3415 | val loss: 0.3427 | val AUPRC: 0.2927
Epoch 15 — train loss: 0.3405 | va